# Preprocessament per classificació (C vs I)

Aquest notebook prepara un dataset únic de característiques per als notebooks de classificació.

Sortides:
- NPZ amb `X`, `y`, `y_detail`, `subj_list` i metadades
- CSV amb resum per subjecte

Un cop executat, els notebooks de classificació només han de carregar aquest artefacte.

In [7]:
import nbformat
from pathlib import Path

# Try to find initialization notebook
paths = [
    Path("./00_inicialització.ipynb"),
    Path("./src/notebooks/00_inicialització.ipynb"),
    Path("/content/drive/My Drive/Universitat/TFG/src/notebooks/00_inicialització.ipynb"),
]

nb_path = next((p for p in paths if p.exists()), None)
if not nb_path:
    raise FileNotFoundError(f"00_inicialització.ipynb not found. Colab: mount Google Drive first.")

# Load and execute initialization
with open(nb_path, encoding="utf-8") as f:
    nb = nbformat.read(f, as_version=4)

for cell in nb.cells:
    if cell.cell_type == "code":
        exec(cell.source, globals())

print("✓ Inicialització carregada.")

FileNotFoundError: 00_inicialització.ipynb not found. Colab: mount Google Drive first.

In [ ]:
def extract_subject_features(fif_path):
    ep = mne.read_epochs(fif_path, preload=True, verbose=False)
    for ch in ep.info["chs"]:
        ch["unit"] = mne.io.constants.FIFF.FIFF_UNIT_V
    ep.set_channel_types({ch: "eeg" for ch in ep.ch_names}, on_unit_change="ignore")

    required_channels = [TARGET_CHANNELS["theta"], TARGET_CHANNELS["alpha"]]
    missing = [ch for ch in required_channels if ch not in ep.ch_names]
    if missing:
        return None, f"{fif_path.name}: falten canals {missing}"

    ep = ep.copy().pick(required_channels).crop(tmin=TARGET_WINDOW[0], tmax=TARGET_WINDOW[1])

    c_names = [n for n in COND_EVENTS["Congruent"] if n in ep.event_id]
    pur_names = [n for n in COND_EVENTS["Incongruent_Pur"] if n in ep.event_id]
    mixt_names = [n for n in COND_EVENTS["Incongruent_Mixt"] if n in ep.event_id]
    i_names = [n for n in COND_EVENTS["Incongruent"] if n in ep.event_id]

    if not c_names or not i_names:
        return None, f"{fif_path.name}: sense trials C o I"

    selected_names = c_names + i_names
    ep_bin = ep[selected_names]

    c_codes = [ep.event_id[n] for n in c_names]
    pur_codes = [ep.event_id[n] for n in pur_names]
    mixt_codes = [ep.event_id[n] for n in mixt_names]

    events = ep_bin.events[:, 2]
    y_detail = np.full(events.shape[0], "Incongruent", dtype=object)
    y_detail[np.isin(events, c_codes)] = "Congruent"
    y_detail[np.isin(events, pur_codes)] = "Incongruent_Pur"
    y_detail[np.isin(events, mixt_codes)] = "Incongruent_Mixt"
    y = np.where(y_detail == "Congruent", 0, 1)

    x_data = ep_bin.get_data().astype(np.float32, copy=False)
    sfreq = ep_bin.info["sfreq"]
    nperseg = int(round(STFT_WINDOW_SEC * sfreq))
    hop = int(round(STFT_STEP_SEC * sfreq))
    noverlap = max(0, nperseg - hop)
    nfft = max(nperseg, int(round(sfreq)))

    f, t, z = stft(
        x_data,
        fs=sfreq,
        window="hann",
        nperseg=nperseg,
        noverlap=noverlap,
        nfft=nfft,
        boundary="zeros",
        padded=True,
        axis=-1,
    )
    power = np.abs(z) ** 2

    theta_idx = np.array([np.argmin(np.abs(f - fr)) for fr in FREQS_THETA])
    alpha_idx = np.array([np.argmin(np.abs(f - fr)) for fr in FREQS_ALPHA])

    theta_power = power[:, 0:1, theta_idx, :]
    alpha_power = power[:, 1:2, alpha_idx, :]

    X = np.concatenate(
        [theta_power.reshape(theta_power.shape[0], -1), alpha_power.reshape(alpha_power.shape[0], -1)],
        axis=1,
    )

    subject_summary = {
        "subject": fif_path.stem,
        "n_trials": int(len(y)),
        "n_c": int(np.sum(y == 0)),
        "n_i": int(np.sum(y == 1)),
        "n_ip": int(np.sum(y_detail == "Incongruent_Pur")),
        "n_im": int(np.sum(y_detail == "Incongruent_Mixt")),
        "n_features": int(X.shape[1]),
    }

    out = {
        "X": X,
        "y": y,
        "y_detail": y_detail,
        "subject": fif_path.stem,
        "summary": subject_summary,
    }
    return out, None

In [ ]:
X_blocks, y_blocks, y_detail_blocks, subj_blocks = [], [], [], []
subject_rows = []
skipped = []

for fif_path in DATA_FILES:
    out, err = extract_subject_features(fif_path)
    if err is not None:
        skipped.append(err)
        continue

    X_blocks.append(out["X"])
    y_blocks.append(out["y"])
    y_detail_blocks.append(out["y_detail"])
    subj_blocks.extend([out["subject"]] * len(out["y"]))
    subject_rows.append(out["summary"])

if not X_blocks:
    raise RuntimeError("No s'han pogut extreure característiques de cap subjecte.")

X = np.vstack(X_blocks)
y = np.concatenate(y_blocks)
y_detail = np.concatenate(y_detail_blocks).astype(object)
subj_list = np.array(subj_blocks, dtype=object)

ICA_COMPONENTS = min(ICA_MAX_COMPONENTS, X.shape[1])

df_subject_summary = pd.DataFrame(subject_rows).sort_values("subject").reset_index(drop=True)

print(f"Mostres totals: {len(y)}")
print(f"C: {(y == 0).sum()} | I: {(y == 1).sum()}")
print(f"Subjectes inclosos: {len(np.unique(subj_list))}")
print(f"Dimensió features: {X.shape}")
print(f"ICA_COMPONENTS recomanat (màxim): {ICA_COMPONENTS}")
if skipped:
    print("Subjectes omesos:")
    for msg in skipped:
        print(f"  - {msg}")

In [ ]:
processed_dir = DATA_DIR / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

dataset_path = processed_dir / "c_vs_i_fcz_theta_oz_alpha_stft.npz"
summary_path = processed_dir / "c_vs_i_subject_summary.csv"

np.savez_compressed(
    dataset_path,
    X=X.astype(np.float32),
    y=y.astype(np.int8),
    y_detail=y_detail.astype(str),
    subj_list=subj_list.astype(str),
    freqs_theta=FREQS_THETA.astype(np.int16),
    freqs_alpha=FREQS_ALPHA.astype(np.int16),
    target_window=np.array(TARGET_WINDOW, dtype=np.float32),
    stft_window_sec=np.float32(STFT_WINDOW_SEC),
    stft_step_sec=np.float32(STFT_STEP_SEC),
    ica_components=np.int16(ICA_COMPONENTS),
    feature_channels=np.array([TARGET_CHANNELS["theta"], TARGET_CHANNELS["alpha"]], dtype=object),
)

df_subject_summary.to_csv(summary_path, index=False)

print(f"Dataset desat a: {dataset_path}")
print(f"Resum per subjecte desat a: {summary_path}")

In [ ]:
display(df_subject_summary)

class_counts = {
    "Congruent": int(np.sum(y_detail == "Congruent")),
    "Incongruent": int(np.sum(y_detail != "Congruent")),
    "Incongruent_Pur": int(np.sum(y_detail == "Incongruent_Pur")),
    "Incongruent_Mixt": int(np.sum(y_detail == "Incongruent_Mixt")),
}

plt.figure(figsize=(8, 4))
plt.bar(class_counts.keys(), class_counts.values(), color=["steelblue", "darkorange", "tomato", "seagreen"])
plt.xticks(rotation=15)
plt.ylabel("Nombre de trials")
plt.title("Distribució de classes després de preprocessament")
plt.tight_layout()
plt.show()